# GT Keyword Extraction and Processing Workflow

In [ ]:
# Configuration
dataset_name = "kotiliesi"
# url_gt = "GT.txt"
# url = "ULR.txt"
dataset_language = "finnish"
base_url = f"https://cs.uef.fi/~himat/WebRank/dataset_12/dataset_12/{dataset_name}"
num_pages = 100

# Stemming, Compound Analysis, and Stopword Libraries
from nltk.stem.snowball import SnowballStemmer
import nltk
from libvoikko import Voikko  # Finnish compound word analysis
from nltk.corpus import stopwords  # Stopword library

# Initialize stemmer, compound analyzer, and stopwords
stemmer = SnowballStemmer(dataset_language)
voikko_analyzer = Voikko("fi")  # For Finnish compound splitting
nltk_finnish_stopwords = set(stopwords.words(dataset_language))

In [12]:
import re
def process_keywords(keywords):
    """Apply cleaning, compound splitting, stemming, and stopword filtering to a list of keywords. Returns a processed list of useful words."""
    processed = []
    for word in keywords:
        # Clean word: keep only alphabetic characters (including Finnish letters)
        cleaned_word = re.sub(r'[^a-zA-ZäöåÄÖÅ]', '', word)
        if not cleaned_word:
            continue
        # Compound splitting (Finnish only, using libvoikko)
        compound_parts = [cleaned_word]
        try:
            analyses = voikko_analyzer.analyze(cleaned_word)
            if analyses:
                best = analyses[0]
                # Add baseform if available
                baseform = best.get('BASEFORM')
                if baseform and baseform.lower() != cleaned_word.lower():
                    compound_parts.append(baseform.lower())
                # Add compound parts if available
                wordbases = best.get('WORDBASES', '')
                if wordbases:
                    parts = [p.lower().strip() for p in wordbases.replace(')', '').split('(') if p]
                    compound_parts.extend(parts)
        except Exception:
            pass
        # Clean all compound parts
        cleaned_parts = [re.sub(r'[^a-zA-ZäöåÄÖÅ]', '', part) for part in compound_parts]
        # Remove empty strings after cleaning
        cleaned_parts = [part for part in cleaned_parts if part]
        # Stem all parts
        stemmed_parts = [stemmer.stem(part) for part in cleaned_parts]
        # Filter stopwords
        filtered = [part for part in stemmed_parts if part not in nltk_finnish_stopwords]
        processed.extend(filtered)
    # Return unique processed keywords (useful words only)
    return processed

In [13]:
from bs4 import BeautifulSoup
def extract_and_process_keywords_from_tag(html_text, tag_name):
    """Extracts keywords from the content inside the specified tag in the given HTML text, processes them using process_keywords, and returns a dictionary with keyword frequencies."""
    soup = BeautifulSoup(html_text, 'html.parser')
    elements = soup.find_all(tag_name)
    raw_keywords = []
    for elem in elements:
        text = elem.get_text(separator=' ')
        raw_keywords.extend(text.split())
    processed = process_keywords(raw_keywords)
    freq = {}
    for kw in processed:
        freq[kw] = freq.get(kw, 0) + 1
    return freq

In [14]:
from bs4 import BeautifulSoup

def extract_available_tags(html_text):
    """
    Extracts and returns a list of unique HTML tag names present in the given html_text.
    """
    soup = BeautifulSoup(html_text, "html.parser")
    tags = set([tag.name for tag in soup.find_all()])
    return sorted(tags)

In [17]:
# 5. Aggregate and Print URL Ratings Across All Pages (as a pseudo-tag for sorting)
from urllib.parse import urlparse, unquote
from collections import defaultdict
import urllib.request

tag_rating_sum = defaultdict(float)
tag_count = defaultdict(int)

for i in range(num_pages):
    gt_url = f"{base_url}/{i}/GT.txt"
    web_page_url = f"{base_url}/{i}"
    url_file_url = f"{base_url}/{i}/URL.txt"
    processed_keywords = []
    try:
        with urllib.request.urlopen(gt_url, timeout=5) as response:
            gt_text = response.read().decode("utf-8-sig").strip()
            gt_keywords = gt_text.split()
            processed_keywords = list(set(process_keywords(gt_keywords)))
    except Exception as e:
        print(f"Error processing GT for page {i}: {e}")
        continue
    # HTML tag ratings
    try:
        with urllib.request.urlopen(web_page_url, timeout=5) as web_response:
            html_text = web_response.read().decode("utf-8-sig").strip()
            tags = extract_available_tags(html_text)
            for tag in tags:
                result = extract_and_process_keywords_from_tag(html_text, tag)
                total_keywords = sum(result.values())
                matched_sum = sum(freq for kw, freq in result.items() if kw in processed_keywords)
                rating = matched_sum / total_keywords if total_keywords > 0 else 0
                tag_rating_sum[tag] += rating
                tag_count[tag] += 1
    except Exception as e:
        print(f"Error processing web page for page {i}: {e}")
        continue
    # URL rating as a pseudo-tag
    try:
        with urllib.request.urlopen(url_file_url, timeout=5) as url_response:
            real_url = url_response.read().decode("utf-8-sig").strip()
            parsed_url = urlparse(real_url)
            normalized_path = unquote(parsed_url.path.lower())
            url_tokens = re.findall(r"[a-zåäöA-ZÅÄÖ0-9]+", normalized_path)
            processed_url_keywords = process_keywords(url_tokens)
            total_url_keywords = len(processed_url_keywords)
            matched_url_keywords = sum(1 for kw in processed_url_keywords if kw in processed_keywords)
            rating = matched_url_keywords / total_url_keywords if total_url_keywords > 0 else 0
            tag_rating_sum['URL'] += rating
            tag_count['URL'] += 1
    except Exception as e:
        print(f"Error processing URL for page {i}: {e}")
        continue

# Calculate sum rating per tag and sort (including URL as a tag)
tag_sum_rating = {tag: tag_rating_sum[tag] for tag in tag_rating_sum}
sorted_tags = sorted(tag_sum_rating.items(), key=lambda x: x[1], reverse=True)

print("Tag and URL rating sum across all pages (sorted):")
for tag, rating_sum in sorted_tags:
    print(f"{tag}: {rating_sum:.4f}")

Error processing GT for page 0: HTTP Error 404: Not Found
Error processing GT for page 1: HTTP Error 404: Not Found
Error processing GT for page 2: HTTP Error 404: Not Found
Error processing GT for page 3: HTTP Error 404: Not Found
Error processing GT for page 4: HTTP Error 404: Not Found
Error processing GT for page 5: HTTP Error 404: Not Found
Error processing GT for page 6: HTTP Error 404: Not Found
Error processing GT for page 7: HTTP Error 404: Not Found
Error processing GT for page 8: HTTP Error 404: Not Found
Error processing GT for page 9: HTTP Error 404: Not Found
Error processing GT for page 10: HTTP Error 404: Not Found
Error processing GT for page 11: HTTP Error 404: Not Found
Error processing GT for page 12: HTTP Error 404: Not Found
Error processing GT for page 13: HTTP Error 404: Not Found
Error processing GT for page 14: HTTP Error 404: Not Found
Error processing GT for page 15: HTTP Error 404: Not Found
Error processing GT for page 16: HTTP Error 404: Not Found
Error p

KeyboardInterrupt: 